# OPSD gap su GSM8K

Misura il gap student vs self-teacher (il *predittore* della legge di Tufa Labs)
su matematica invece che coding. Nessun training.

**Runtime -> Cambia tipo di runtime -> GPU (T4).**

In [ ]:
!nvidia-smi

In [ ]:
# 1. repo + dipendenze (~3-5 min)
REPO = "https://github.com/FRANCESCO-USERNAME/opsd-gap-gsm8k.git"  # <-- cambia
!git clone -q $REPO /content/opsd-gap-gsm8k || echo 'gia clonato'
%cd /content/opsd-gap-gsm8k
!pip install -q vllm datasets

## A. Che aspetto hanno i problemi di GSM8K

In [ ]:
from datasets import load_dataset
import textwrap

ds = load_dataset("openai/gsm8k", "main", split="test")
print("problemi nel test set:", len(ds))

for i in range(3):
    r = ds[i]
    print("=" * 78)
    print("PROBLEMA %d" % i)
    print("=" * 78)
    print(textwrap.fill(r["question"], 76))
    print("\n--- soluzione di riferimento (annotata a mano) ---")
    print(r["answer"])
    print()

In [ ]:
# quanto sono lunghi i problemi e quante operazioni servono
import numpy as np

sub = ds.select(range(200))
words = [len(r["question"].split()) for r in sub]
steps = [r["answer"].count("<<") for r in sub]  # <<48/2=24>> = un calcolo annotato
print("parole per problema: mediana %d  (min %d, max %d)" % (
    np.median(words), min(words), max(words)))
print("calcoli per soluzione: mediana %d  (min %d, max %d)" % (
    np.median(steps), min(steps), max(steps)))
print("distribuzione dei calcoli:", {k: steps.count(k) for k in sorted(set(steps))})

## B. Che aspetto hanno i prompt del self-teacher

Esempi sintetici, senza GPU. Serve a controllare che nessun contesto
privilegiato riveli la risposta finale (answer leakage).

In [ ]:
!python src/inspect_prompts.py

## C. Smoke test

In [ ]:
# 10 problemi, 4 campioni, 3 condizioni (~5 min)
!python src/run_eval.py --n-problems 10 --n-samples 4 --tag smoke \
    --conditions none,feedback_binary,gt_solution

In [ ]:
# prompt REALI usati nello smoke test, cosi come li ha visti il modello
import json

ex = json.load(open("results/smoke_prompt_examples.json", encoding="utf-8"))
for cond, items in ex.items():
    if not items:
        print("(%s: nessun campione con contesto)\n" % cond)
        continue
    print("#" * 78)
    print("# CONDIZIONE: %s   (problema %d, campione %d)" % (
        cond, items[0]["problem"], items[0]["sample"]))
    print("#" * 78)
    print(items[0]["prompt"])
    print()

## D. Run completo e analisi

In [ ]:
# tutte le condizioni (lungo - partire da 100 problemi)
!python src/run_eval.py --n-problems 100 --n-samples 8 --tag run

In [ ]:
# esempi di prompt del run completo, una condizione alla volta
import json

ex = json.load(open("results/run_prompt_examples.json", encoding="utf-8"))
COND = "feedback_diag"  # <-- cambia per vedere le altre
for item in ex[COND]:
    print("-" * 78)
    print(item["prompt"])
    print()

In [ ]:
# tabella + grafico
!python src/analyze.py --tag run
from IPython.display import Image
Image('figures/gaps.png')

In [ ]:
# scarica i risultati da ricommittare nel repo
from google.colab import files
!zip -qr results.zip results figures
files.download('results.zip')